In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import image, datasets, plotting
from nilearn.input_data import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure
from scipy import stats
import seaborn as sns

# Define where the data is stored
data_dir = 'data/ds000051'  # Folder with your data

# Load the brain atlas
atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-1mm')
atlas_filename = atlas['filename']
roi_names = atlas['labels']

# Set up the masker with the atlas
masker = NiftiLabelsMasker(labels_img=atlas_filename, standardize=True, detrend=True)

# Set up the connectivity measure
connectivity_measure = ConnectivityMeasure(kind='correlation')

# Initialize a list to store the connectivity results
connectivity_results = []

# Function to get time series data for a specific trial type
def get_trial_data(time_series, events_df, condition):
    trial_events = events_df[events_df['trial_type'].str.contains(condition)]
    trial_data = []
    for _, event in trial_events.iterrows():
        start = int(event['onset'] / 2)  # Assuming TR = 2s
        end = start + int(event['duration'] / 2)
        trial_data.append(time_series[start:end])
    return np.concatenate(trial_data), len(trial_events)

# Loop over all participants and runs
for participant in range(1, 14):  # Participants are numbered from 1 to 13
    participant_id = f'sub-{participant:02d}'
    for run in range(1, 9):  # Runs are numbered from 1 to 8
        bold_file = os.path.join(data_dir, f'{participant_id}/func/{participant_id}_task-abstractconcretejudgment_run-{run:02d}_bold.nii.gz')
        events_file = os.path.join(data_dir, f'{participant_id}/func/{participant_id}_task-abstractconcretejudgment_run-{run:02d}_events.tsv')
        
        if not os.path.exists(bold_file) or not os.path.exists(events_file):
            continue  # Skip if either file does not exist
        
        # Load the functional MRI image
        func_img = image.load_img(bold_file)
        
        # Extract time series data for each brain region
        time_series = masker.fit_transform(func_img)
        
        # Load the events file
        events_df = pd.read_csv(events_file, sep='\t')
        
        # Calculate connectivity for different trial types
        trial_types = ['Spanish', 'English', 'Abstract', 'Concrete', 'Novel', 'Repeat',
                       'Spanish-Spanish', 'English-English', 'Spanish-English', 'English-Spanish']
        
        for trial_type in trial_types:
            trial_data, trial_count = get_trial_data(time_series, events_df, trial_type)
            connectivity_matrix = connectivity_measure.fit_transform([trial_data])[0]
            
            # Calculate the average connectivity value
            triu_indices = np.triu_indices_from(connectivity_matrix, k=1)
            upper_triangle_values = connectivity_matrix[triu_indices]
            average_connectivity_value = np.mean(upper_triangle_values)
            
            # Store the results
            connectivity_results.append({
                'participant': participant,
                'run': run,
                'trial_type': trial_type,
                'average_connectivity_value': average_connectivity_value,
                'trial_count': trial_count
            })

# Convert the results to a DataFrame
connectivity_df = pd.DataFrame(connectivity_results)
print(connectivity_df)

# Ensure that the ROI names match the size of the connectivity matrix
matrix_shape = connectivity_matrix.shape[0]
if len(roi_names) > matrix_shape:
    roi_names = roi_names[:matrix_shape]
elif len(roi_names) < matrix_shape:
    raise ValueError("Number of regions in atlas is less than the connectivity matrix dimensions.")

# Plot the connectivity matrix for the last participant, run, and trial type
plt.figure(figsize=(10, 8))
plotting.plot_matrix(connectivity_matrix, figure=(10, 8), labels=roi_names, vmax=1, vmin=-1, reorder=True, 
                     title=f'ROI-to-ROI Connectivity Matrix: {participant_id} Run-{run:02d} {trial_type}')
plt.show()

# Calculate and print the overall average connectivity value
overall_average_connectivity_value = np.mean(connectivity_df['average_connectivity_value'])
print(f"Overall Average Brain Connectivity Value: {overall_average_connectivity_value}")

# Analysis of connectivity differences between conditions
plt.figure(figsize=(15, 5 * 3))

for i, condition in enumerate(['language', 'concreteness', 'novelty']):
    if condition == 'language':
        condition1, condition2 = 'Spanish', 'English'
    elif condition == 'concreteness':
        condition1, condition2 = 'Abstract', 'Concrete'
    else:  # novelty
        condition1, condition2 = 'Novel', 'Repeat'
    
    condition1_connectivity = connectivity_df[connectivity_df['trial_type'] == condition1]['average_connectivity_value']
    condition2_connectivity = connectivity_df[connectivity_df['trial_type'] == condition2]['average_connectivity_value']
    
    t_stat, p_value = stats.ttest_ind(condition1_connectivity, condition2_connectivity)
    
    print(f"\n{condition.capitalize()} Difference ({condition1} vs {condition2}):")
    print(f"t-statistic: {t_stat}")
    print(f"p-value: {p_value}")

    # Create subplot for each condition
    plt.subplot(3, 2, 2*i + 1)
    sns.boxplot(x='trial_type', y='average_connectivity_value', 
                data=connectivity_df[connectivity_df['trial_type'].isin([condition1, condition2])])
    plt.title(f'Box Plot: {condition.capitalize()} Comparison')
    plt.ylabel('Average Connectivity Value')

    plt.subplot(3, 2, 2*i + 2)
    sns.violinplot(x='trial_type', y='average_connectivity_value', 
                   data=connectivity_df[connectivity_df['trial_type'].isin([condition1, condition2])])
    plt.title(f'Violin Plot: {condition.capitalize()} Comparison')
    plt.ylabel('Average Connectivity Value')

plt.tight_layout()
plt.show()

# Additional overall comparison
plt.figure(figsize=(12, 6))
sns.boxplot(x='trial_type', y='average_connectivity_value', data=connectivity_df)
plt.title('Average Connectivity Values by Trial Type')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Compare trial counts
trial_counts = connectivity_df.groupby('trial_type')['trial_count'].mean()
print("\nAverage trial counts per run:")
print(trial_counts)

# Compare first language vs second language
first_lang = connectivity_df[connectivity_df['trial_type'].isin(['Spanish-Spanish', 'English-English'])]
second_lang = connectivity_df[connectivity_df['trial_type'].isin(['Spanish-English', 'English-Spanish'])]

t_stat, p_value = stats.ttest_ind(first_lang['average_connectivity_value'], 
                                  second_lang['average_connectivity_value'])
print("\nFirst Language vs Second Language:")
print(f"t-statistic: {t_stat}")
print(f"p-value: {p_value}")

# Compare within-language vs across-language trials
within_lang = connectivity_df[connectivity_df['trial_type'].isin(['Spanish-Spanish', 'English-English'])]
across_lang = connectivity_df[connectivity_df['trial_type'].isin(['Spanish-English', 'English-Spanish'])]

t_stat, p_value = stats.ttest_ind(within_lang['average_connectivity_value'], 
                                  across_lang['average_connectivity_value'])
print("\nWithin-Language vs Across-Language:")
print(f"t-statistic: {t_stat}")
print(f"p-value: {p_value}")

# Visualize these comparisons
plt.figure(figsize=(12, 6))
sns.boxplot(x='trial_type', y='average_connectivity_value', 
            data=connectivity_df[connectivity_df['trial_type'].isin(['Spanish-Spanish', 'English-English', 'Spanish-English', 'English-Spanish'])])
plt.title('Connectivity Values: Within-Language vs Across-Language')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
!pip install plotly


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import image, datasets, plotting
from nilearn.input_data import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure
from scipy import stats
import seaborn as sns

# Define where the data is stored
data_dir = 'data/ds000051'  # Folder with your data

# Load the brain atlas
atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-1mm')
atlas_filename = atlas['filename']
roi_names = atlas['labels']

# Set up the masker with the atlas
masker = NiftiLabelsMasker(labels_img=atlas_filename, standardize=True, detrend=True)

# Set up the connectivity measure
connectivity_measure = ConnectivityMeasure(kind='correlation')

# Initialize a list to store the connectivity results
connectivity_results = []

# Function to get time series data for a specific trial type
def get_trial_data(time_series, events_df, condition):
    trial_events = events_df[events_df['trial_type'].str.contains(condition)]
    trial_data = []
    for _, event in trial_events.iterrows():
        start = int(event['onset'] / 2)  # Assuming TR = 2s
        end = start + int(event['duration'] / 2)
        trial_data.append(time_series[start:end])
    return np.concatenate(trial_data), len(trial_events)

# Loop over all participants and runs
for participant in range(1, 14):  # Participants are numbered from 1 to 13
    participant_id = f'sub-{participant:02d}'
    for run in range(1, 9):  # Runs are numbered from 1 to 8
        bold_file = os.path.join(data_dir, f'{participant_id}/func/{participant_id}_task-abstractconcretejudgment_run-{run:02d}_bold.nii.gz')
        events_file = os.path.join(data_dir, f'{participant_id}/func/{participant_id}_task-abstractconcretejudgment_run-{run:02d}_events.tsv')
        
        if not os.path.exists(bold_file) or not os.path.exists(events_file):
            continue  # Skip if either file does not exist
        
        # Load the functional MRI image
        func_img = image.load_img(bold_file)
        
        # Extract time series data for each brain region
        time_series = masker.fit_transform(func_img)
        
        # Load the events file
        events_df = pd.read_csv(events_file, sep='\t')
        
        # Calculate connectivity for different trial types
        trial_types = ['Spanish', 'English', 'Abstract', 'Concrete', 'Novel', 'Repeat',
                       'Spanish-Spanish', 'English-English', 'Spanish-English', 'English-Spanish']
        
        for trial_type in trial_types:
            trial_data, trial_count = get_trial_data(time_series, events_df, trial_type)
            connectivity_matrix = connectivity_measure.fit_transform([trial_data])[0]
            
            # Calculate the average connectivity value
            triu_indices = np.triu_indices_from(connectivity_matrix, k=1)
            upper_triangle_values = connectivity_matrix[triu_indices]
            average_connectivity_value = np.mean(upper_triangle_values)
            
            # Store the results
            connectivity_results.append({
                'participant': participant,
                'run': run,
                'trial_type': trial_type,
                'average_connectivity_value': average_connectivity_value,
                'trial_count': trial_count
            })

# Convert the results to a DataFrame
connectivity_df = pd.DataFrame(connectivity_results)
print(connectivity_df)

# Ensure that the ROI names match the size of the connectivity matrix
matrix_shape = connectivity_matrix.shape[0]
if len(roi_names) > matrix_shape:
    roi_names = roi_names[:matrix_shape]
elif len(roi_names) < matrix_shape:
    raise ValueError("Number of regions in atlas is less than the connectivity matrix dimensions.")

# Plot the connectivity matrix for the last participant, run, and trial type
plt.figure(figsize=(10, 8))
plotting.plot_matrix(connectivity_matrix, figure=(10, 8), labels=roi_names, vmax=1, vmin=-1, reorder=True, 
                     title=f'ROI-to-ROI Connectivity Matrix: {participant_id} Run-{run:02d} {trial_type}')
plt.show()

# Calculate and print the overall average connectivity value
overall_average_connectivity_value = np.mean(connectivity_df['average_connectivity_value'])
print(f"Overall Average Brain Connectivity Value: {overall_average_connectivity_value}")

# Analysis of connectivity differences between conditions
plt.figure(figsize=(15, 5 * 3))

for i, condition in enumerate(['language', 'concreteness', 'novelty']):
    if condition == 'language':
        condition1, condition2 = 'Spanish', 'English'
    elif condition == 'concreteness':
        condition1, condition2 = 'Abstract', 'Concrete'
    else:  # novelty
        condition1, condition2 = 'Novel', 'Repeat'
    
    condition1_connectivity = connectivity_df[connectivity_df['trial_type'] == condition1]['average_connectivity_value']
    condition2_connectivity = connectivity_df[connectivity_df['trial_type'] == condition2]['average_connectivity_value']
    
    t_stat, p_value = stats.ttest_ind(condition1_connectivity, condition2_connectivity)
    
    print(f"\n{condition.capitalize()} Difference ({condition1} vs {condition2}):")
    print(f"t-statistic: {t_stat}")
    print(f"p-value: {p_value}")

    # Create subplot for each condition
    plt.subplot(3, 2, 2*i + 1)
    sns.boxplot(x='trial_type', y='average_connectivity_value', 
                data=connectivity_df[connectivity_df['trial_type'].isin([condition1, condition2])])
    plt.title(f'Box Plot: {condition.capitalize()} Comparison')
    plt.ylabel('Average Connectivity Value')

    plt.subplot(3, 2, 2*i + 2)
    sns.violinplot(x='trial_type', y='average_connectivity_value', 
                   data=connectivity_df[connectivity_df['trial_type'].isin([condition1, condition2])])
    plt.title(f'Violin Plot: {condition.capitalize()} Comparison')
    plt.ylabel('Average Connectivity Value')

plt.tight_layout()
plt.show()

# Additional overall comparison
plt.figure(figsize=(12, 6))
sns.boxplot(x='trial_type', y='average_connectivity_value', data=connectivity_df)
plt.title('Average Connectivity Values by Trial Type')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Compare trial counts
trial_counts = connectivity_df.groupby('trial_type')['trial_count'].mean()
print("\nAverage trial counts per run:")
print(trial_counts)

# Compare first language vs second language
first_lang = connectivity_df[connectivity_df['trial_type'].isin(['Spanish-Spanish', 'English-English'])]
second_lang = connectivity_df[connectivity_df['trial_type'].isin(['Spanish-English', 'English-Spanish'])]

t_stat, p_value = stats.ttest_ind(first_lang['average_connectivity_value'], 
                                  second_lang['average_connectivity_value'])
print("\nFirst Language vs Second Language:")
print(f"t-statistic: {t_stat}")
print(f"p-value: {p_value}")

# Compare within-language vs across-language trials
within_lang = connectivity_df[connectivity_df['trial_type'].isin(['Spanish-Spanish', 'English-English'])]
across_lang = connectivity_df[connectivity_df['trial_type'].isin(['Spanish-English', 'English-Spanish'])]

t_stat, p_value = stats.ttest_ind(within_lang['average_connectivity_value'], 
                                  across_lang['average_connectivity_value'])
print("\nWithin-Language vs Across-Language:")
print(f"t-statistic: {t_stat}")
print(f"p-value: {p_value}")

# Visualize these comparisons
plt.figure(figsize=(12, 6))
sns.boxplot(x='trial_type', y='average_connectivity_value', 
            data=connectivity_df[connectivity_df['trial_type'].isin(['Spanish-Spanish', 'English-English', 'Spanish-English', 'English-Spanish'])])
plt.title('Connectivity Values: Within-Language vs Across-Language')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()